In [1]:
import folium
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

In [2]:
# ---------------------------
# 1) Generate random sensor locations in Cali
# ---------------------------

# Approximate bounding box for Cali, Colombia (lat/lon)
# These are rough bounding coordinates around Cali
min_lat, max_lat = 3.34, 3.48
min_lon, max_lon = -76.60, -76.45

# Number of sensors
num_sensors = 10

# Randomly generate sensor coordinates and store in a DataFrame
np.random.seed(42)  # For reproducibility
sensor_data = pd.DataFrame({
    'sensor_id': range(num_sensors),
    'latitude': np.random.uniform(min_lat, max_lat, num_sensors),
    'longitude': np.random.uniform(min_lon, max_lon, num_sensors)
})

# ---------------------------
# 2) Create a time series for 24 hours, with data points every 5 minutes
# ---------------------------
start_time = datetime(2025, 1, 1, 0, 0, 0)
end_time = start_time + timedelta(hours=24)
time_interval = timedelta(minutes=5)  # or 10 if you prefer
timestamps = []
current_time = start_time

while current_time <= end_time:
    timestamps.append(current_time)
    current_time += time_interval

# ---------------------------
# 3) Simulate environmental variables for each sensor over time
# ---------------------------
# For simplicity, let's simulate:
# - CO2 (ppm)
# - Temperature (°C)
# - Humidity (%)

# We will create one row per (sensor, time).
# That can be large, so be mindful. For 24h * 5-min intervals = 288 steps * 10 sensors = 2880 rows

records = []
for t in timestamps:
    for i in range(num_sensors):
        # You can create more realistic simulations, e.g. daily cycles with sine waves + noise
        co2_val = 400 + 100 * np.sin((t.hour / 24.0) * 2 * np.pi) + np.random.normal(0, 20)
        temp_val = 20 + 5 * np.sin((t.hour / 24.0) * 2 * np.pi) + np.random.normal(0, 1)
        hum_val  = 60 + 10 * np.cos((t.hour / 24.0) * 2 * np.pi) + np.random.normal(0, 5)
        
        records.append({
            'datetime': t,
            'sensor_id': i,
            'CO2': co2_val,
            'Temperature': temp_val,
            'Humidity': hum_val
        })

sensor_measurements = pd.DataFrame(records)

# ---------------------------
# 4) Function to create a Folium map for a given time
# ---------------------------
def create_map_for_time(time_point, 
                        sensor_info_df, 
                        sensor_vals_df, 
                        output_folder='maps'):
    """
    Creates a Folium map of Cali, with sensor markers sized or colored
    by CO2 (or another variable) at a specific time_point, and saves to HTML.
    """
    # Filter sensor measurements at that time
    df_current = sensor_vals_df[sensor_vals_df['datetime'] == time_point]
    
    # Merge with sensor locations
    df_merged = pd.merge(sensor_info_df, df_current, on='sensor_id')
    
    # Create base map centered on Cali
    # Approx center of Cali: 3.4516° N, 76.5320° W
    folium_map = folium.Map(location=[3.4516, -76.5320], zoom_start=12)

    # Add sensor markers
    for _, row in df_merged.iterrows():
        co2_level = row['CO2']
        # We will color-code the markers by CO2 level:
        # lower than 400 -> green, 400-600 -> orange, above 600 -> red (for example)
        if co2_level < 400:
            color = 'green'
        elif co2_level < 600:
            color = 'orange'
        else:
            color = 'red'
        
        popup_text = (f"<b>Sensor ID:</b> {row['sensor_id']}<br>"
                      f"<b>CO₂:</b> {row['CO2']:.2f} ppm<br>"
                      f"<b>Temp:</b> {row['Temperature']:.2f} °C<br>"
                      f"<b>Humidity:</b> {row['Humidity']:.2f} %<br>"
                      f"<b>Time:</b> {time_point}")

        folium.CircleMarker(location=[row['latitude'], row['longitude']],
                            radius=5, 
                            color=color,
                            fill=True,
                            fill_color=color,
                            popup=popup_text).add_to(folium_map)

    # Save the map
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    map_filename = f"map_{time_point.strftime('%Y%m%d_%H%M')}.html"
    folium_map.save(os.path.join(output_folder, map_filename))
    print(f"Saved: {map_filename}")

In [3]:
# ---------------------------
# 5) Loop over each timestamp and create/save map
# ---------------------------
for t in timestamps:
    create_map_for_time(t, sensor_data, sensor_measurements)

Saved: map_20250101_0000.html
Saved: map_20250101_0005.html
Saved: map_20250101_0010.html
Saved: map_20250101_0015.html
Saved: map_20250101_0020.html
Saved: map_20250101_0025.html
Saved: map_20250101_0030.html
Saved: map_20250101_0035.html
Saved: map_20250101_0040.html
Saved: map_20250101_0045.html
Saved: map_20250101_0050.html
Saved: map_20250101_0055.html
Saved: map_20250101_0100.html
Saved: map_20250101_0105.html
Saved: map_20250101_0110.html
Saved: map_20250101_0115.html
Saved: map_20250101_0120.html
Saved: map_20250101_0125.html
Saved: map_20250101_0130.html
Saved: map_20250101_0135.html
Saved: map_20250101_0140.html
Saved: map_20250101_0145.html
Saved: map_20250101_0150.html
Saved: map_20250101_0155.html
Saved: map_20250101_0200.html
Saved: map_20250101_0205.html
Saved: map_20250101_0210.html
Saved: map_20250101_0215.html
Saved: map_20250101_0220.html
Saved: map_20250101_0225.html
Saved: map_20250101_0230.html
Saved: map_20250101_0235.html
Saved: map_20250101_0240.html
Saved: map

Saved: map_20250101_2300.html
Saved: map_20250101_2305.html
Saved: map_20250101_2310.html
Saved: map_20250101_2315.html
Saved: map_20250101_2320.html
Saved: map_20250101_2325.html
Saved: map_20250101_2330.html
Saved: map_20250101_2335.html
Saved: map_20250101_2340.html
Saved: map_20250101_2345.html
Saved: map_20250101_2350.html
Saved: map_20250101_2355.html
Saved: map_20250102_0000.html


In [4]:

def simulate_and_display_map():
    """
    Creates a single Folium map of 'sensors' around Cali with random CO2 values,
    and returns the folium.Map object so that it displays in the notebook.
    """

    # -------------------------------------------------------------------------
    # Step 1: Generate random sensor locations and data
    # -------------------------------------------------------------------------
    
    # Approx bounding box for Cali
    min_lat, max_lat = 3.34, 3.48
    min_lon, max_lon = -76.60, -76.45

    # Number of sensors
    num_sensors = 10
    np.random.seed(42)
    
    # Generate random coordinates
    sensor_lats = np.random.uniform(min_lat, max_lat, num_sensors)
    sensor_lons = np.random.uniform(min_lon, max_lon, num_sensors)
    
    # Simulate some random CO2 values
    co2_values = np.random.normal(loc=450, scale=50, size=num_sensors)  
    
    # -------------------------------------------------------------------------
    # Step 2: Create a Folium map centered on Cali
    # -------------------------------------------------------------------------
    cali_center = [3.4516, -76.5320]  # Approx center of Cali
    folium_map = folium.Map(location=cali_center, zoom_start=12)
    
    # -------------------------------------------------------------------------
    # Step 3: Add markers to the map
    # -------------------------------------------------------------------------
    for i in range(num_sensors):
        co2 = co2_values[i]
        
        # Define a simple color scheme based on CO2
        if co2 < 400:
            color = 'green'
        elif co2 < 600:
            color = 'orange'
        else:
            color = 'red'
        
        popup_text = f"Sensor {i}<br>CO2: {co2:.2f} ppm"
        
        folium.CircleMarker(
            location=[sensor_lats[i], sensor_lons[i]],
            radius=5,
            color=color,
            fill=True,
            fill_color=color,
            popup=popup_text
        ).add_to(folium_map)

    # -------------------------------------------------------------------------
    # Step 4: Return the map to display it inline in Jupyter
    # -------------------------------------------------------------------------
    return folium_map

# In a Jupyter Notebook cell, calling the function will display the map inline:
simulate_and_display_map()


In [5]:
import folium
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from ipywidgets import interact, IntSlider
from IPython.display import display

# Create a list of timestamps (e.g., every 2 hours in a day)
start_time = datetime(2025, 1, 1, 0, 0)
time_steps = [start_time + timedelta(hours=i*2) for i in range(12)]  # 12 steps, every 2 hours

# Some sample data: random sensor locations
min_lat, max_lat = 3.34, 3.48
min_lon, max_lon = -76.60, -76.45
num_sensors = 5
np.random.seed(42)
sensor_lats = np.random.uniform(min_lat, max_lat, num_sensors)
sensor_lons = np.random.uniform(min_lon, max_lon, num_sensors)

# Generate random CO2 for each sensor at each time
all_data = {}
for t in time_steps:
    # For each time, simulate some CO2
    co2_values = np.random.normal(loc=450, scale=50, size=num_sensors)
    # Store them in a dict keyed by datetime
    all_data[t] = co2_values

def create_map_for_timestamp(t_idx):
    """
    Given an index into 'time_steps', create and return a folium map
    showing CO2 at that time.
    """
    t = time_steps[t_idx]
    co2_values = all_data[t]
    
    # Create map centered in Cali
    folium_map = folium.Map(location=[3.4516, -76.5320], zoom_start=12)
    
    # Add markers
    for i in range(num_sensors):
        co2 = co2_values[i]
        # Simple color code
        if co2 < 400:
            color = 'green'
        elif co2 < 600:
            color = 'orange'
        else:
            color = 'red'

        popup_text = (
            f"<b>Sensor {i}</b><br>"
            f"Time: {t}<br>"
            f"CO2: {co2:.1f} ppm"
        )
        folium.CircleMarker(
            location=(sensor_lats[i], sensor_lons[i]),
            radius=5,
            color=color,
            fill=True,
            fill_opacity=0.7,
            fill_color=color,
            popup=popup_text
        ).add_to(folium_map)
    return folium_map

def show_map_with_time_slider():
    # Create an interactive slider for time index
    slider = IntSlider(min=0, max=len(time_steps)-1, step=1, value=0, description='Time Index')

    @interact(time_idx=slider)
    def display_map(time_idx):
        # Each time the slider changes, we re-generate the map for that time
        folium_map = create_map_for_timestamp(time_idx)
        display(folium_map)

# Call this function in a Jupyter cell to get the slider + map
show_map_with_time_slider()


interactive(children=(IntSlider(value=0, description='Time Index', max=11), Output()), _dom_classes=('widget-i…